In [ ]:
import pandas as pd

MISSING = ["NA", "PS", "NULL", "PrivacySuppressed"]
inst = pd.read_csv("../data/raw/Most-Recent-Cohorts-Institution.csv", dtype=str, keep_default_na=False)

scope = inst[(inst["CURROPER"] == "1")
             & (inst["DISTANCEONLY"] != "1")
             & (inst["CONTROL"].isin(["1", "2"]))
             & (inst["PREDDEG"].isin(["1", "2", "3"]))
             & (inst["HIGHDEG"].isin(["2", "3", "4"]))
             & (inst["UGDS"] != "0")].copy()

keep = ["UNITID", "INSTNM", "CITY", "STABBR", "CONTROL", "PREDDEG", "HIGHDEG",
        "INSTURL", "NPCURL",
        "OPENADMP", "ADM_RATE", "ADMCON7",
        "SATVR25", "SATVR75", "SATMT25", "SATMT75", "ACTCM25", "ACTCM75",
        "COSTT4_A", "TUITIONFEE_IN", "TUITIONFEE_OUT",
        "LOCALE", "REGION", "UGDS", "LATITUDE", "LONGITUDE","C150_4", "C150_L4"]
colleges = scope[keep].copy()

# Net price: publics use the _PUB column, privates the _PRIV column
public = scope["CONTROL"] == "1"
colleges["NPT4"] = scope["NPT4_PUB"].where(public, scope["NPT4_PRIV"])
for i in range(1, 6):
    colleges[f"NPT4{i}"] = scope[f"NPT4{i}_PUB"].where(public, scope[f"NPT4{i}_PRIV"])

# One missing convention: empty string
colleges = colleges.replace(MISSING, "")

# Clean URLs
def clean_url(u):
    u = u.strip()
    if u == "":
        return u
    if not u.lower().startswith(("http://", "https://")):
        u = "https://" + u
    return u.replace(" ", "%20")

for c in ["INSTURL", "NPCURL"]:
    colleges[c] = colleges[c].map(clean_url)

colleges.to_csv("../data/processed/colleges.csv", index=False)
print(colleges.shape)

(2989, 32)


In [3]:
fos = pd.read_csv("../data/raw/Most-Recent-Cohorts-Field-of-Study.csv", dtype=str, keep_default_na=False,
                  usecols=["UNITID", "CIPCODE", "CIPDESC", "CREDLEV", "IPEDSCOUNT1", "IPEDSCOUNT2"])

programs = fos[(fos["UNITID"] != "NA")
               & (fos["CREDLEV"].isin(["2", "3"]))
               & (fos["UNITID"].isin(colleges["UNITID"]))].replace(MISSING, "")

programs.to_csv("../data/processed/programs.csv", index=False)
print(programs.shape, programs["UNITID"].nunique())

(105248, 6) 2960


In [4]:
check = pd.read_csv("../data/processed/colleges.csv", dtype=str, keep_default_na=False)
print(check.shape)
print("Leftover NA/PS:", check.isin(["NA", "PS"]).sum().sum())
print(check[check["INSTNM"] == "University of Tulsa"][["NPT4", "NPT41", "NPCURL"]].T)

(2989, 32)
Leftover NA/PS: 0
                                            1916
NPT4                                       15000
NPT41                                       5578
NPCURL  https://utulsa.edu/net-price-calculator/


In [5]:
special = ["CUNY Bronx Community College", "CUNY Lehman College", "University of Tulsa",
           "CUNY Hunter College", "University of Mount Saint Vincent",
           "Belanger School of Nursing", "Moorpark College",
           "Southern New Hampshire University", "PENSACOLA CHRISTIAN COLLEGE", "Harvard University"]

picked = colleges[colleges["INSTNM"].isin(special)]
others = colleges[~colleges["INSTNM"].isin(special)].sample(20, random_state=42)
sample = pd.concat([picked, others])

sample.to_csv("../data/processed/colleges_sample.csv", index=False)
programs[programs["UNITID"].isin(sample["UNITID"])].to_csv("../data/processed/programs_sample.csv", index=False)
print(len(sample))

30
